# Gold: lab_test_trends

Builds `adb_amnuay.gold.lab_test_trends` (one row per test type per calendar month) from `silver.lab_results`, enriched with calendar month and quarter names, plus month-over-month abnormal rate trend via LAG(). Full rebuild every run, per STTM `sttm_silver_to_gold.xlsx`.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_amnuay.gold.lab_test_trends
COMMENT 'Gold lab test trends: one row per test_name per calendar month with volumes, abnormal/critical rates and month-over-month abnormal rate change. Source: STTM silver to gold.'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                           AS year,
    month(test_date)                          AS month,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count
  FROM adb_amnuay.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(try_divide(m.abnormal_test_count, m.total_tests) * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(try_divide(m.critical_test_count, m.total_tests) * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  CAST(w.test_name AS STRING)                                       AS test_name,
  CAST(w.year AS INT)                                               AS year,
  CAST(w.month AS INT)                                              AS month,
  CAST(c.month_name AS STRING)                                      AS month_name,
  CAST(c.quarter_name AS STRING)                                    AS quarter_name,
  CAST(w.total_tests AS INT)                                        AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                AS critical_test_count,
  w.abnormal_rate_pct                                               AS abnormal_rate_pct,
  w.critical_rate_pct                                               AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))              AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN adb_amnuay.silver.calendar c
  ON c.date_key = w.year * 10000 + w.month * 100 + 1

In [0]:
# Return the number of records in the rebuilt gold table to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table("adb_amnuay.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))